In [14]:
from collections import defaultdict
import gymnasium as gym
import numpy as np
import race_car   # registers "RaceCar-v0"

env = gym.make("RaceCar-v0")
state_size = env.observation_space.shape[0]  # Size of the state space
action_size = env.action_space.n  # Number of possible actions

## Actor/Critic Networks

In RL, the Actor-Critic architecture is commonly used to train agents. The Actor network is responsible for selecting actions based on the current game state, and the Critic is responsible for scoring those actions. The two networks are trained together and improve each other over time.

This Actor network is a simple three-layer feedforward neural network that takes in the current game state and outputs the probabilities of each possible action.

In [15]:
from torch.distributions import Categorical
import torch.nn as nn
import torch.nn.functional as F

class Actor(nn.Module):
    def __init__(self, state_size, action_size):
        super(Actor, self).__init__()
        self.state_size = state_size
        self.action_size = action_size
        self.linear1 = nn.Linear(self.state_size, 128)
        self.linear2 = nn.Linear(128, 256)
        self.linear3 = nn.Linear(256, self.action_size)

    def forward(self, state):
        l1_res = F.relu(self.linear1(state))
        l2_res = F.relu(self.linear2(l1_res))
        logits = self.linear3(l2_res)
        # we use categorical distribution to help wrap our probability distributions in some helpful tools that we'll use more later
        distribution = Categorical(logits=logits)
        return distribution

In [16]:
class Critic(nn.Module):
    def __init__(self, state_size):
        super(Critic, self).__init__()
        self.state_size = state_size
        self.linear1 = nn.Linear(self.state_size, 128)
        self.linear2 = nn.Linear(128, 256)
        self.linear3 = nn.Linear(256, 1)

    def forward(self, state):
        l1_res = F.relu(self.linear1(state))
        l2_res = F.relu(self.linear2(l1_res))
        l3_res = self.linear3(l2_res)
        return l3_res

The Critic network is nearly identical, but it outputs a single value--the expected value of the state.

## PPO Agent

I used Generalized Advantage Estimation (GAE) to compute the advantage function. GAE is the method used in the PPO paper. GAE takes in the realized rewards for each step, the predicted rewards for each step, and some hyperparameters. Then, it calculates the difference between the realized and predicted rewards (with some decay). The delta is then used to compute the GAE, which is the delta plus the discounted GAE from the next step. Think of the GAE as the carry-through reward for the next step; if the next step is better than expected, then the current step is also better than expected.

Knowing what the decay on the delta is isn't hugely important to PPO, but a good mental model is that the further in the future some reward is, the less likely the current step is to be responsible for it. Your being able to cut a straight line in preschool probably has less of a bearing on whether you will get into MIT. From a math standpoint, we add the next step's GAE * gamma * lam. Gamma is the discount rate for future rewards (think about cutting and MIT), and lambda (lam) is the decay for the GAE. Lambda determines much a model should trust it's own predictions; 0 means it trusts itself a lot, and 1 means it doesn't trust itself at all. The PPO paper uses 0.95, which is a good balance between trusting the model and trusting the environment.

In [17]:
def compute_gae(next_value, rewards, masks, values, gamma=0.99, lam=0.95):
    advantages = torch.zeros((num_steps, num_envs))
    returns = torch.zeros((num_steps, num_envs))
    gae = 0
    for step in reversed(range(len(rewards))):
        nv = next_value if step == len(rewards) - 1 else values[step + 1]
        delta = rewards[step] + gamma * nv * masks[step] - values[step]
        # gae adds a "carry-through" reward to advantages, which is weighted
        # basically: how surprising was the next step, and how much should we trust that surprise?
        gae = delta + gamma * lam * masks[step] * gae
        advantages[step] = gae
        # compute the returns for each step in the rollout, which is the advantage plus the value
        returns[step] = advantages[step] + values[step]
    return advantages, returns

In [18]:
batch_size = 2048
minibatch_size = 512
update_epochs = 4
clip_eps = 0.2

In [19]:
import gymnasium
import torch

num_envs = 16
num_steps = 128

env = gymnasium.make_vec(
    "RaceCar-v0",
    num_envs=num_envs,
    vectorization_mode="sync",
    vector_kwargs={"autoreset_mode": gymnasium.vector.AutoresetMode.SAME_STEP},
)
# store the observations, actions, logprobs, rewards, dones, and values for each step in the rollout
obs_dim = env.single_observation_space.shape[0]
obs_buf      = torch.zeros((num_steps, num_envs, obs_dim))
actions_buf  = torch.zeros((num_steps, num_envs))
logprobs_buf = torch.zeros((num_steps, num_envs))
rewards_buf  = torch.zeros((num_steps, num_envs))
dones_buf    = torch.zeros((num_steps, num_envs))
values_buf   = torch.zeros((num_steps, num_envs))

# initialize the actor/critic networks
actor = Actor(obs_dim, env.single_action_space.n)
critic = Critic(obs_dim)

next_obs, _ = env.reset()
next_obs = torch.tensor(next_obs, dtype=torch.float32)

def collect_rollout():
    global next_obs

    for step in range(num_steps):
        obs_buf[step] = next_obs

        with torch.no_grad():
            distribution = actor(next_obs)
            action = distribution.sample()
            logprob = distribution.log_prob(action)
            value = torch.squeeze(critic(next_obs), dim=-1)

        actions_buf[step] = action
        logprobs_buf[step] = logprob
        values_buf[step] = value

        # act in the environment and then get the next observation based on our predicted action
        next_obs, reward, done, truncated, info = env.step(action.numpy())

        rewards_buf[step] = torch.tensor(reward, dtype=torch.float32)
        dones_buf[step] = torch.tensor(done | truncated, dtype=torch.float32)

        next_obs = torch.tensor(next_obs, dtype=torch.float32)

    with torch.no_grad():
        next_value = torch.squeeze(critic(next_obs), dim=-1)

    return next_value

In [20]:
gae_buf = torch.zeros((num_steps, num_envs))
next_value = collect_rollout()

# compute the GAE and returns for each step in the rollout
gae_buf, returns_buf = compute_gae(next_value, rewards_buf, 1 - dones_buf, values_buf)

In [21]:
optimizer = torch.optim.Adam(list(actor.parameters()) + list(critic.parameters()), lr=3e-4)

In [22]:
def ppo_update(gae_buf, returns_buf):
    b_obs = obs_buf.reshape(num_steps * num_envs, obs_dim)
    b_gae = gae_buf.flatten()
    b_logprobs = logprobs_buf.flatten()
    b_returns = returns_buf.flatten()
    b_actions = actions_buf.flatten()

    for epoch in range(update_epochs):
        minibatch_order = torch.randperm(num_steps * num_envs)
        for start in range(0, num_steps * num_envs, minibatch_size):
            end = start + minibatch_size
            minibatch_indices = minibatch_order[start:end]

            m_obs = b_obs[minibatch_indices]
            m_gae = b_gae[minibatch_indices]
            m_logprobs = b_logprobs[minibatch_indices]
            m_returns = b_returns[minibatch_indices]
            m_actions = b_actions[minibatch_indices]

            dist = actor(m_obs)
            new_logprobs = dist.log_prob(m_actions)

            new_values = torch.squeeze(critic(m_obs), dim=-1)

            ratio = torch.exp(new_logprobs - m_logprobs)

            # normalize the minibatch advantages
            m_gae = (m_gae - m_gae.mean()) / (m_gae.std() + 1e-8)
            policy_loss = -torch.mean(torch.min(ratio * m_gae, torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * m_gae))
            value_loss = torch.mean((new_values - m_returns) ** 2)
            total_loss = policy_loss + 0.5 * value_loss - 0.01 * dist.entropy().mean()

            # now backprop
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
        # output the loss for this epoch and progress
        print(f"Epoch {epoch + 1}/{update_epochs}, Loss: {total_loss.item():.4f}")

In [25]:
import copy
import numpy as np
import imageio
from PIL import Image as PILImage, ImageDraw
from IPython.display import Video, display

# Separate env just for watching. Never step your training envs here.
eval_env = gymnasium.make("RaceCar-v0", render_mode="rgb_array")

def record_episode(actor, seed=42, max_steps=1000):
    """Play one episode with the most likely action each step; return the frames."""
    frames = []
    obs, _ = eval_env.reset(seed=seed)   # same seed → every checkpoint faces the same start
    for _ in range(max_steps):
        frames.append(eval_env.render())
        with torch.no_grad():
            dist = actor(torch.tensor(obs, dtype=torch.float32))
            action = dist.probs.argmax().item()      # deterministic for demos
        obs, reward, terminated, truncated, _ = eval_env.step(action)
        if terminated or truncated:
            frames.append(eval_env.render())
            break
    return frames

In [ ]:
# now we can train the agent using PPO
num_updates = 400

checkpoint_every = 20
checkpoints = {0: copy.deepcopy(actor.state_dict())}

for update in range(num_updates):
    next_value = collect_rollout()
    gae_buf, returns_buf = compute_gae(next_value, rewards_buf, 1 - dones_buf, values_buf)
    ppo_update(gae_buf, returns_buf)
    episodes = int(dones_buf.sum().item())
    # a rollout can finish zero episodes (e.g. the lander hovers), so don't divide by zero
    avg_reward = rewards_buf.sum().item() / max(episodes, 1)
    print(f"Update {update + 1}/{max(num_updates, 1)} complete. Episodes: {episodes}, Average Reward: {avg_reward:.2f}")

    if (update + 1) % checkpoint_every == 0:
        checkpoints[update + 1] = copy.deepcopy(actor.state_dict())
        torch.save(actor.state_dict(), f"actor_update_{update + 1}.pt")

Epoch 1/4, Loss: 0.0308
Epoch 2/4, Loss: 0.0272
Epoch 3/4, Loss: 0.0434
Epoch 4/4, Loss: 0.0373
Update 1/400 complete. Episodes: 4, Average Reward: 48.75
Epoch 1/4, Loss: 0.0387
Epoch 2/4, Loss: 0.0339
Epoch 3/4, Loss: 0.0325
Epoch 4/4, Loss: 0.0402
Update 2/400 complete. Episodes: 5, Average Reward: 41.34
Epoch 1/4, Loss: 0.0567
Epoch 2/4, Loss: 0.0486
Epoch 3/4, Loss: 0.0726
Epoch 4/4, Loss: 0.0528
Update 3/400 complete. Episodes: 10, Average Reward: 25.27
Epoch 1/4, Loss: 0.0196
Epoch 2/4, Loss: 0.0205
Epoch 3/4, Loss: 0.0142
Epoch 4/4, Loss: 0.0087
Update 4/400 complete. Episodes: 2, Average Reward: 87.27
Epoch 1/4, Loss: 0.3701
Epoch 2/4, Loss: 0.3547
Epoch 3/4, Loss: 0.1856
Epoch 4/4, Loss: 0.1825
Update 5/400 complete. Episodes: 10, Average Reward: 23.75
Epoch 1/4, Loss: 0.1635
Epoch 2/4, Loss: 0.1389
Epoch 3/4, Loss: 0.0824
Epoch 4/4, Loss: 0.0716
Update 6/400 complete. Episodes: 5, Average Reward: 41.15
Epoch 1/4, Loss: 0.0243
Epoch 2/4, Loss: 0.0163
Epoch 3/4, Loss: 0.0124
Ep

In [ ]:
def load_actor(state_dict):
    a = Actor(obs_dim, env.single_action_space.n)   # match your constructor's arguments
    a.load_state_dict(state_dict)
    a.eval()
    return a

def add_label(frame, text):
    img = PILImage.fromarray(frame)
    ImageDraw.Draw(img).text((10, 10), text, fill=(255, 255, 255))
    return np.array(img)

def side_by_side(updates_to_show, fps=20, path="progress.mp4"):
    clips = []
    for u in updates_to_show:
        frames = record_episode(load_actor(checkpoints[u]))
        clips.append([add_label(f, f"update {u}  (lasted {len(frames)-1} steps)") for f in frames])

    # Hold each clip's last frame so short episodes "freeze" while longer ones keep going
    longest = max(len(c) for c in clips)
    clips = [c + [c[-1]] * (longest - len(c)) for c in clips]

    combined = [np.concatenate([c[i] for c in clips], axis=1) for i in range(longest)]
    imageio.mimsave(path, combined, fps=fps)
    display(Video(path, embed=True))

side_by_side([0, 20, 60, max(checkpoints)])